# Chimeric receptors simulations - Assertion testing

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
        
import maboss
from maboss import temporal_logic as tl

### Sanity test (apoptosis and prolif cannot happen at the same time)

In [2]:
bnd = "model_Tcell.bnd"
cfg = "model_Tcell.cfg"

nodes = list(maboss.load(bnd,cfg).network.keys())
#print(nodes)

ON = [0,1]
OFF = [1,0]
RANDOM = [0.5, 0.5]

all_nodes_random_init = [{'node':node,'istate':RANDOM} for node in nodes]
#print(all_nodes_off)

# initial conditions
i_c_TME = [
    {'node':'LCK','istate':ON},
    {'node':'CD45','istate':ON},
    {'node':'CD8','istate':ON},
    {'node':'TIM3_L','istate':ON},
    {'node':'IFNG_L','istate':ON},
    {'node':'IL2_L','istate':ON},
    {'node':'IL4_L','istate':ON},
    {'node':'TGFB_L','istate':ON},
    {'node':'L41BB','istate':ON},
    {'node':'FAS_L','istate':ON},
    {'node':'TIGIT_L','istate':ON},
    {'node':'PD1_L','istate':ON},
    {'node':'IL7_L','istate':ON},
    {'node':'CD80_86','istate':ON},
    {'node':'IL12_L','istate':ON},
    {'node':'IL27_L','istate':ON},
    {'node':'ICOS_L','istate':ON},
]

i_c_MM = [
    {'node':'LCK','istate':ON},
    {'node':'CD45','istate':ON},
    {'node':'CD8','istate':ON},
    {'node':'TIM3_L','istate':ON},
    {'node':'IFNG_L','istate':ON},
    {'node':'IL2_L','istate':ON},
    {'node':'IL4_L','istate':ON},
    {'node':'TGFB_L','istate':ON},
    {'node':'L41BB','istate':ON},
    {'node':'FAS_L','istate':ON},
    {'node':'TIGIT_L','istate':ON},
    {'node':'PD1_L','istate':ON}
]

i_c_antigen = [
    {'node':'LCK','istate':ON},
    {'node':'CD45','istate':ON},
    {'node':'CD8','istate':ON}
]

# CD4
# initial states to have types of cells
CD4 = all_nodes_random_init + [{'node':'iden_CD4','istate':ON}]
CD4_endo = CD4 + [{'node': 'iden_abTCR', 'istate':ON}]
CD4_is_CART =  CD4 + i_c_TME + [{'node':'iden_CAR','istate':ON}]
CD4_is_Tcell = [{'node':'iden_abTCR','istate':ON}] + CD4
CD4_is_TEG001 = [{'node':'iden_gdTCR','istate':ON},] + CD4
CD4_is_TEG103_trunc = [{'node':'iden_TR103','istate':ON},{'node':'iden_gdTCR','istate':ON}] + CD4
CD4_is_NKG2D_CD28 = [{'node':'iden_NKCD28','istate':ON}, {'node':'iden_gdTCR','istate':ON}] + CD4
CD4_is_NKG2D_41BB = [{'node':'iden_NK41BB','istate':ON}, {'node':'iden_gdTCR','istate':ON}] + CD4

#CD8
CD8 = all_nodes_random_init + [{'node':'iden_CD8','istate':ON}]
CD8_endo = CD8 + [{'node': 'iden_abTCR', 'istate':ON}]

th1_phenotype_ic = [
    {'node': 'IL12_L', 'istate': OFF},
    {'node': 'IFNG_L', 'istate': ON},
    {'node': 'IL4_L', 'istate': ON}
]

th2_phenotype_ic = [
    {'node': 'IL12_L', 'istate': OFF},
    {'node': 'IFNG_L', 'istate': OFF},
    {'node': 'IL4_L', 'istate': ON}
]

th1_phenotype_mut = "IL12_L:OFF IFNG_L:ON IL4_L:ON"
th2_phenotype_mut = "IL12_L:OFF IFNG_L:OFF IL4_L:ON"

### Run the simulations and the querying
if you want to use 2 initial conditions use `list1.extend(list2)`
**eg: CART with TME : `tl.MaBoSSEvaluator.querying(assertions,cfg,bnd,is_CART.extend(i_c_TME)`**

- assertion 2: th1 is a node, look for output th1, mutation IL-12 ->1 IL-4 -> 0
- assertion 3: same for th2
- assertion 4: same assertion 1 but different outputs
- assertion 5: EOMES node

In [3]:
CD4_endo_list_phenotypes = [
    f"Inc(node:Th1) / [ ] [ {th1_phenotype_mut} ] [ ]", #CD4 T cells acquire a Th1 phenotype under IFNG_L, IL12_L, and under no IL4_L.
    f"Inc(node:Th2) / [ ] [ {th2_phenotype_mut} ] [ ]", #CD4 T cells acquire a Th2 phenotype under IL4_L and no IFNG_L nor IL12_L
]

CD4_endo_list_random_ic = [
    f"Inc(node:GZMB,PRF1,IFNg,TBET,RUNX3,ThPOK,EOMES) / [ ] [ {th1_phenotype_mut} ] [ ]", #CD4 T cells under IFNG_L, IL12_L, and under no IL4_L, increase of GZMB, PRF1, IFNg, TBET, RUNX3, ThPOK and EOMES
    f"Inc(node:GZMB,PRF1,IFNg,TBET,RUNX3,ThPOK,EOMES) / [ ] [ {th1_phenotype_mut} ] [ comb ]"
    f"Inc(node:EOMES) / [ ] [ {th2_phenotype_mut} ] [ ]", #CD4 T cells under IL4_L and no IFNG_L nor IL12_L increase EOMES
    f"Dec(node:PRF1,GZMB,GNLY,IFNg) / [ ] [ RUNX3:OFF ] [ ]", #RUNX3 KO results in PRF1, GZMB and GNLY decrease and has no effect in IFNg.
    f"Dec(node:PRF1,GZMB,GNLY,IFNg) / [ ] [ RUNX3:OFF ] [ comb ]"
    f"Dec(node:PRF1,GZMB,TNFa,GZMK,IFNg,GNLY) / [ ] [ TBET:OFF ] [ ]", # TBET KO results in decrease of PRF1, GZMB (but not as much as RUNX3 KO), TNFa, GZMK, IFNg and GNLY.
    f"Dec(node:PRF1,GZMB,TNFa,GZMK,IFNg,GNLY) / [ ] [ TBET:OFF ] [ comb ]",
    f"Inc(node:PRF1,GZMB) / [ ] [ EOMES:OFF ] [ ]", #EOMES KO has no effect on PRF1 or GZMB
    f"Inc(node:PRF1,GZMB) / [ ] [ ThPOK:OFF ] [ ]", #ThPOK KO increased PRF1 and GZMB
    f"Inc(node:PRF1,GZMB) / [ ] [ EOMES:OFF ] [ comb ]", #EOMES KO has no effect on PRF1 or GZMB with comb
    f"Inc(node:PRF1,GZMB) / [ ] [ ThPOK:OFF ] [ comb ]", #ThPOK KO increased PRF1 and GZMB with comb
    #int41BB KI AND CD3 KI decrease Apoptosis to a lower level than CD3 KI or int41BB KI
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:OFF CD3:ON ] [ ]",
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect.
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ ]", #should show stable
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect. with comb
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ comb ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ comb ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ comb ]", #should show stable
    #int41BB KI AND CD3 KI increase IL2 higher than int41BB KI
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    # and int41BB KI does so higher than CD3 KI
    f"Inc(node:IL2) / [ ] [ CD3:ON int41BB:OFF ] [ ]",
    # IL7_L decreases apoptosis
    f"Dec(node:Apoptosis) / [ ] [ IL7_L:ON ] [ ]",
    #IL2_L or IFNG_L increase TBET, BLIMP1, EOMES, GZMB and PRF1
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IL2_L:ON IFNG_L:OFF ] [ ]",
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IFNG_L:ON IL2_L:OFF ] [ ]",
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IFNG_L:ON IL2_L:ON ] [ ]",
    #IL2_L or IFNG_L increase TBET, BLIMP1, EOMES, GZMB and PRF1 with comb
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IL2_L:ON IFNG_L:OFF ] [ comb ]",
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IFNG_L:ON IL2_L:OFF ] [ comb ]",
    f"Inc(node:TBET,BLIMP1,EOMES,GZMB,PRF1) / [ ] [ IFNG_L:ON IL2_L:ON ] [ comb ]",
    #TGFB_L and IL2_L increase Treg, FOXP3, IL2RA and IL10
    f"Inc(node:Treg,FOXP3,IL2RA,IL10) / [ ] [ IL2_L:ON TGFB_L:ON ] [ ]",
    #TGFB_L and IL2_L increase Treg, FOXP3, IL2RA and IL10 with comb
    f"Inc(node:Treg,FOXP3,IL2RA,IL10) / [ ] [ IL2_L:ON TGFB_L:ON ] [ comb ]",
]
output_CD4_endo_list_random_ic = ["GZMB","PRF1","IFNg","TBET","RUNX3","ThPOK","EOMES","GNLY","TNFa","GZMK","Apoptosis","proliferation","IL2","BLIMP1","Treg","FOXP3","IL2RA","IL10"]

CD4_endo_list_th1_ic = [
    f"Inc(node:GZMB,PRF1,IFNg,TBET,RUNX3,ThPOK,EOMES) / [ ] [ IFNG_L:ON IL12_L:ON IL4_L:OFF ] [ ]",#CD4 T cells under IFNG_L, IL12_L, and under no IL4_L, increase of GZMB, PRF1, IFNg, TBET, RUNX3, ThPOK and EOMES
    f"Inc(node:GZMB,PRF1,IFNg,TBET,RUNX3,ThPOK,EOMES) / [ ] [ IFNG_L:ON IL12_L:ON IL4_L:OFF ] [ comb ]",#CD4 T cells under IFNG_L, IL12_L, and under no IL4_L, increase of GZMB, PRF1, IFNg, TBET, RUNX3, ThPOK and EOMES with comb
    f"Dec(node:PRF1,GZMB,GNLY,IFNg) / [ ] [ RUNX3:OFF ] [ ]", #RUNX3 KO results in PRF1, GZMB and GNLY decrease and has no effect in IFNg.
    f"Dec(node:PRF1,GZMB,GNLY,IFNg) / [ ] [ RUNX3:OFF ] [ comb ]", #RUNX3 KO results in PRF1, GZMB and GNLY decrease and has no effect in IFNg with comb
    f"Dec(node:PRF1,GZMB,TNFa,GZMK,IFNg,GNLY) / [ ] [ TBET:OFF ] [ ]", # TBET KO results in decrease of PRF1, GZMB (but not as much as RUNX3 KO), TNFa, GZMK, IFNg and GNLY.
    f"Dec(node:PRF1,GZMB,TNFa,GZMK,IFNg,GNLY) / [ ] [ TBET:OFF ] [ comb ]", # TBET KO results in decrease of PRF1, GZMB (but not as much as RUNX3 KO), TNFa, GZMK, IFNg and GNLY.
    f"Inc(node:PRF1,GZMB) / [ ] [ EOMES:OFF ] [ ]", #EOMES KO has no effect on PRF1 or GZMB
    f"Inc(node:PRF1,GZMB) / [ ] [ ThPOK:OFF ] [ ]", #ThPOK KO increased PRF1 and GZMB
    f"Inc(node:PRF1,GZMB) / [ ] [ EOMES:OFF ] [ comb ]", #EOMES KO has no effect on PRF1 or GZMB with comb
    f"Inc(node:PRF1,GZMB) / [ ] [ ThPOK:OFF ] [ comb ]", #ThPOK KO increased PRF1 and GZMB with comb
]
output_CD4_endo_list_th1 = ["GZMB","PRF1","IFNg","TBET","RUNX3","ThPOK","EOMES","GNLY","TNFa","GZMK"]

CD4_endo_list_th2_ic = [
    f"Inc(node:EOMES) / [ ] [ IL4_L:ON IFNG_L:ON IL12_L:ON ] [ ]", #CD4 T cells under IL4_L and no IFNG_L nor IL12_L increase EOMES
]

CD4_endo_list_abTCR_L41BB = [
    #int41BB KI AND CD3 KI decrease Apoptosis to a lower level than CD3 KI or int41BB KI
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:OFF CD3:ON ] [ ]",
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect.
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ ]", #should show stable
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect. with comb
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ comb ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ comb ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ comb ]", #should show stable
    #int41BB KI AND CD3 KI increase IL2 higher than int41BB KI
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    # and int41BB KI does so higher than CD3 KI
    f"Inc(node:IL2) / [ ] [ CD3:ON int41BB:OFF ] [ ]",
]
output_CD4_endo_list_abTCR_L41BB = ["Apoptosis","proliferation","IL2","IFNg","GZMB","GZMK","PRF1"]

CD4_endo_CAR41BB_i_c = [
    #THEMIS KO or SHP1 KO increase Apoptosis
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:ON ] [ ]",
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:OFF SHP1:ON ] [ ]",
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:OFF ] [ ]",
]

CD8_list_random_ic = [
    #int41BB KI AND CD3 KI decrease Apoptosis to a lower level than CD3 KI or int41BB KI
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:OFF CD3:ON ] [ ]",
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect.
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ ]", #should show stable
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect. with comb
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ comb ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ comb ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ comb ]", #should show stable
    #int41BB KI AND CD3 KI increase IL2 higher than int41BB KI
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    # and int41BB KI does so higher than CD3 KI
    f"Inc(node:IL2) / [ ] [ CD3:ON int41BB:OFF ] [ ]",
    # IL7_L decreases apoptosis
    f"Dec(node:Apoptosis) / [ ] [ IL7_L:ON ] [ ]",
]
output_CD8_list_random_ic = ["Apoptosis","proliferation","IL2","IFNg","GZMB","GZMK","PRF1"]

CD8_list_abTCR_L41BB = [
    #int41BB KI AND CD3 KI decrease Apoptosis to a lower level than CD3 KI or int41BB KI
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    f"Dec(node:Apoptosis) / [ ] [ int41BB:OFF CD3:ON ] [ ]",
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect.
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ ]", #should show stable
    #int41BB KI AND CD3 KI increase proliferation, IFNg, GZMB, GZMK, PRF1, CD3 KI or int41BB KI have no effect. with comb
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:ON ] [ comb ]",
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:ON CD3:OFF ] [ comb ]", #should show stable
    f"Inc(node:proliferation,IFNg,GZMB,GZMK,PRF1) / [ ] [ int41BB:OFF CD3:ON ] [ comb ]", #should show stable
    #int41BB KI AND CD3 KI increase IL2 higher than int41BB KI
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:ON ] [ ]",
    f"Inc(node:IL2) / [ ] [ int41BB:ON CD3:OFF ] [ ]",
    # and int41BB KI does so higher than CD3 KI
    f"Inc(node:IL2) / [ ] [ CD3:ON int41BB:OFF ] [ ]",
]
outptut_CD8_list_abTCR_L41BB = ["Apoptosis","proliferation","IL2","IFNg","GZMB","GZMK","PRF1"]

CD8_CAR41BB_i_c = [
    #THEMIS KO or SHP1 KO increase Apoptosis
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:ON ] [ ]",
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:OFF SHP1:ON ] [ ]",
    f"Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:OFF ] [ ]",
]

## Simulations CD4 endo

In [4]:
res_phenotypes = tl.MaBoSSEvaluator.querying(CD4_endo_list_phenotypes,cfg,bnd,CD4_endo,["Th1","Th2"])
res_random_ic = tl.MaBoSSEvaluator.querying(CD4_endo_list_random_ic,cfg,bnd,CD4_endo,output_CD4_endo_list_random_ic)
res_th1_ic = tl.MaBoSSEvaluator.querying(CD4_endo_list_th1_ic,cfg,bnd,CD4_endo,output_CD4_endo_list_th1)
res_th2_ic = tl.MaBoSSEvaluator.querying(CD4_endo_list_th2_ic,cfg,bnd,CD4_endo,["EOMES"])
res_abTCR_ic = tl.MaBoSSEvaluator.querying(CD4_endo_list_abTCR_L41BB,cfg,bnd,CD4_endo,output_CD4_endo_list_abTCR_L41BB)
res_CAR41BB_ic = tl.MaBoSSEvaluator.querying(CD4_endo_CAR41BB_i_c,cfg,bnd,CD4_endo,["Apoptosis"])

Evaluations done !
Evaluations done !
Evaluations done !
Evaluations done !
Evaluations done !
Evaluations done !


## Simulation CD8

In [5]:
res_CD8_random = tl.MaBoSSEvaluator.querying(CD8_list_random_ic,cfg,bnd,CD8_endo,output_CD8_list_random_ic)
res_CD8_abTCR = tl.MaBoSSEvaluator.querying(CD8_list_abTCR_L41BB,cfg,bnd,CD8_endo,outptut_CD8_list_abTCR_L41BB)
res_CD8_CAR41BB_ic = tl.MaBoSSEvaluator.querying(CD8_CAR41BB_i_c,cfg,bnd,CD8_endo,["Apoptosis"])

Evaluations done !
Evaluations done !
Evaluations done !


### Results diplay

In [6]:
tl.Visualiser(list_queries=CD4_endo_list_phenotypes, list_of_results=res_phenotypes).display_results_and_queries(title="Phenotypes")
tl.Visualiser(list_queries=CD4_endo_list_random_ic, list_of_results=res_random_ic).display_results_and_queries(title="CD4 endogenous Random IC")
tl.Visualiser(list_queries=CD4_endo_list_th1_ic, list_of_results=res_th1_ic).display_results_and_queries(title="CD4 endogenous Th1 IC")
tl.Visualiser(list_queries=CD4_endo_list_th2_ic, list_of_results=res_th2_ic).display_results_and_queries(title="CD4 endogenous Th2 IC")
tl.Visualiser(list_queries=CD4_endo_list_abTCR_L41BB, list_of_results=res_abTCR_ic).display_results_and_queries(title="CD4 endogenous abTCR L41BB IC")
tl.Visualiser(list_queries=CD4_endo_CAR41BB_i_c, list_of_results=res_CAR41BB_ic).display_results_and_queries(title="CD4 endogenous CAR41BB IC")

Phenotypes
Query 1 : Inc(node:Th1) / [ ] [ IL12_L:OFF IFNG_L:ON IL4_L:ON ] [ ]


,Th1 from master,Th1 from mutation,Difference Th1,Percentage Th1,Increase Th1
0,0.3709,0.4104,0.0395,10.65%,True


Query 2 : Inc(node:Th2) / [ ] [ IL12_L:OFF IFNG_L:OFF IL4_L:ON ] [ ]


,Th2 from master,Th2 from mutation,Difference Th2,Percentage Th2,Increase Th2
0,0.0021,0.0051,0.003,142.86%,True


CD4 endogenous Random IC
Query 1 : Inc(node:GZMB,PRF1,IFNg,TBET,RUNX3,ThPOK,EOMES) / [ ] [ IL12_L:OFF IFNG_L:ON IL4_L:ON ] [ ]


...

,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2679,0.0025,0.94%,True


Query 2 : Inc(node:Apoptosis) / [ ] [ THEMIS:OFF SHP1:ON ] [ ]


,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2675,0.0021,0.79%,True


Query 3 : Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:OFF ] [ ]


,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2644,-0.001,-0.38%,False


In [7]:
tl.Visualiser(list_queries=CD8_list_random_ic, list_of_results=res_phenotypes).display_results_and_queries(title="CD8 endogenous Random IC")
tl.Visualiser(list_queries=CD8_list_abTCR_L41BB, list_of_results=res_abTCR_ic).display_results_and_queries(title="CD8 endogenous abTCR L41BB IC")
tl.Visualiser(list_queries=CD8_CAR41BB_i_c, list_of_results=res_CAR41BB_ic).display_results_and_queries(title="CD8 endogenous CAR41BB IC")

CD8 endogenous Random IC
Query 1 : Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]


,Th1 from master,Th1 from mutation,Difference Th1,Percentage Th1,Increase Th1
0,0.3709,0.4104,0.0395,10.65%,True


Query 2 : Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:OFF ] [ ]


,Th2 from master,Th2 from mutation,Difference Th2,Percentage Th2,Increase Th2
0,0.0021,0.0051,0.003,142.86%,True


CD8 endogenous abTCR L41BB IC
Query 1 : Dec(node:Apoptosis) / [ ] [ int41BB:ON CD3:ON ] [ ]


...

,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2679,0.0025,0.94%,True


Query 2 : Inc(node:Apoptosis) / [ ] [ THEMIS:OFF SHP1:ON ] [ ]


,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2675,0.0021,0.79%,True


Query 3 : Inc(node:Apoptosis) / [ ] [ THEMIS:ON SHP1:OFF ] [ ]


,Apoptosis from master,Apoptosis from mutation,Difference Apoptosis,Percentage Apoptosis,Increase Apoptosis
0,0.2654,0.2644,-0.001,-0.38%,False
